# Bank RAG — Qwen3 policy embeddings

This notebook creates the **experimental** 1024-dimensional policy index. Before running, attach the private Dataset that contains `policy-chunks.jsonl`, select a GPU accelerator, and enable Internet. The notebook does not call DeepSeek or any other LLM.

In [ ]:
%pip install -q "sentence-transformers>=5.0,<6.0" "transformers>=4.51,<6.0" "huggingface-hub>=0.30,<2.0" "pyarrow>=18,<24" "pandas>=2.2,<3.0"

In [ ]:
from __future__ import annotations

import gc
import hashlib
import importlib.metadata
import json
import os
from datetime import datetime, timezone
from pathlib import Path
from zipfile import ZipFile

os.environ.setdefault('PYTORCH_ALLOC_CONF', 'expandable_segments:True')

import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
import torch
from huggingface_hub import HfApi
from sentence_transformers import SentenceTransformer

INPUT_ROOT = Path('/kaggle/input')
OUTPUT_ROOT = Path('/kaggle/working')
EXTRACTED_ROOT = OUTPUT_ROOT / 'attached-private-dataset'
REQUIRED_INPUT_FILENAMES = {
    'policy-chunks.jsonl',
    'policy-sources.jsonl',
    'embedding-job.json',
}

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def sha256_text(value: str) -> str:
    return hashlib.sha256(value.encode('utf-8')).hexdigest()

def prepare_input_bundle() -> None:
    if list(INPUT_ROOT.rglob('policy-chunks.jsonl')):
        return
    matching_archives = []
    for zip_path in INPUT_ROOT.rglob('*.zip'):
        with ZipFile(zip_path) as archive:
            if REQUIRED_INPUT_FILENAMES.issubset(set(archive.namelist())):
                matching_archives.append(zip_path)
    if len(matching_archives) != 1:
        raise RuntimeError(
            f'Expected one attached embedding bundle ZIP, found: {matching_archives}'
        )
    EXTRACTED_ROOT.mkdir(parents=True, exist_ok=True)
    with ZipFile(matching_archives[0]) as archive:
        for filename in REQUIRED_INPUT_FILENAMES | {'normalization-report.json'}:
            if filename in archive.namelist():
                (EXTRACTED_ROOT / filename).write_bytes(archive.read(filename))

def find_unique(filename: str) -> Path:
    matches = sorted(INPUT_ROOT.rglob(filename))
    if EXTRACTED_ROOT.is_dir():
        matches.extend(sorted(EXTRACTED_ROOT.rglob(filename)))
    if len(matches) != 1:
        raise RuntimeError(f'Expected exactly one {filename}, found: {matches}')
    return matches[0]

def load_jsonl(path: Path) -> list[dict]:
    with path.open('r', encoding='utf-8') as handle:
        return [json.loads(line) for line in handle if line.strip()]

def package_version(name: str) -> str:
    return importlib.metadata.version(name)


In [ ]:
prepare_input_bundle()
chunks_path = find_unique('policy-chunks.jsonl')
sources_path = find_unique('policy-sources.jsonl')
job_path = find_unique('embedding-job.json')

chunks = load_jsonl(chunks_path)
sources = load_jsonl(sources_path)
job = json.loads(job_path.read_text(encoding='utf-8'))
source_by_id = {item['source_id']: item for item in sources}

if not chunks:
    raise RuntimeError('No chunks were loaded')
chunk_ids = [item['chunk_id'] for item in chunks]
if len(chunk_ids) != len(set(chunk_ids)):
    raise RuntimeError('Duplicate chunk_id values found')
missing_sources = sorted({item['source_id'] for item in chunks} - set(source_by_id))
if missing_sources:
    raise RuntimeError(f'Missing source records: {missing_sources}')

allowed_statuses = set(job['allowed_source_statuses'])
unexpected_statuses = {
    version['status']
    for source in sources
    for version in source['versions']
    if version['status'] not in allowed_statuses
}
if unexpected_statuses:
    raise RuntimeError(f'Unexpected source statuses: {unexpected_statuses}')

MODEL_ID = job['model_id']
REQUESTED_REVISION = job['requested_revision']
DIMENSION = int(job['embedding_dimension'])
# Hard caps keep the job safe on Kaggle's 14-16 GB single-GPU runtimes,
# including when an older attached Dataset still requests 16 x 4096.
BATCH_SIZE = min(int(job['batch_size']), 2)
MAX_SEQUENCE_LENGTH = min(int(job['max_sequence_length']), 3072)
NORMALIZE = bool(job['normalize_embeddings'])
QUERY_INSTRUCTION = job['query_instruction']

print({
    'chunks_path': str(chunks_path),
    'sources_path': str(sources_path),
    'chunks': len(chunks),
    'sources': len(sources),
    'index_tier': job['index_tier'],
})

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model_info = HfApi().model_info(MODEL_ID, revision=REQUESTED_REVISION)
resolved_revision = model_info.sha
if not resolved_revision:
    raise RuntimeError('Could not resolve an immutable Hugging Face revision')

model = SentenceTransformer(
    MODEL_ID,
    revision=resolved_revision,
    device=device,
)
if device == 'cuda':
    model.half()
model.max_seq_length = MAX_SEQUENCE_LENGTH
actual_dimension = model.get_sentence_embedding_dimension()
if actual_dimension != DIMENSION:
    raise RuntimeError(f'Expected dimension {DIMENSION}, model returned {actual_dimension}')

print({
    'device': device,
    'model': MODEL_ID,
    'resolved_revision': resolved_revision,
    'dimension': actual_dimension,
    'max_sequence_length': model.max_seq_length,
})

In [ ]:
embedding_inputs = []
embedding_input_hashes = []
for chunk in chunks:
    source = source_by_id[chunk['source_id']]
    heading_path = ' > '.join(chunk['heading_path'])
    text = job['input_template'].format(
        title=source['title'],
        heading_path=heading_path,
        content=chunk['content'],
    )
    embedding_inputs.append(text)
    embedding_input_hashes.append(f'sha256:{sha256_text(text)}')

vectors = None
used_batch_size = None
for candidate_batch_size in sorted({BATCH_SIZE, 1}, reverse=True):
    try:
        vectors = model.encode(
            embedding_inputs,
            batch_size=candidate_batch_size,
            show_progress_bar=True,
            convert_to_numpy=True,
            normalize_embeddings=NORMALIZE,
        )
        used_batch_size = candidate_batch_size
        break
    except torch.OutOfMemoryError:
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        print(f'CUDA OOM at batch size {candidate_batch_size}; retrying smaller batch.')
if vectors is None or used_batch_size is None:
    raise RuntimeError('Embedding failed at every configured batch size')
vectors = np.asarray(vectors, dtype=np.float32)

if vectors.shape != (len(chunks), DIMENSION):
    raise RuntimeError(f'Unexpected vector shape: {vectors.shape}')
if not np.isfinite(vectors).all():
    raise RuntimeError('Embedding matrix contains NaN or infinite values')

norms = np.linalg.norm(vectors, axis=1)
if NORMALIZE and not np.allclose(norms, 1.0, atol=1e-4):
    raise RuntimeError('Normalized vector norms are outside tolerance')

print({
    'shape': vectors.shape,
    'dtype': str(vectors.dtype),
    'batch_size': used_batch_size,
    'norm_min': float(norms.min()),
    'norm_max': float(norms.max()),
})

In [ ]:
vector_values = pa.array(vectors.reshape(-1), type=pa.float32())
embedding_column = pa.FixedSizeListArray.from_arrays(vector_values, DIMENSION)

table = pa.table({
    'chunk_id': [item['chunk_id'] for item in chunks],
    'source_id': [item['source_id'] for item in chunks],
    'version_id': [item['version_id'] for item in chunks],
    'namespace': [item['namespace'] for item in chunks],
    'chunk_index': [item['chunk_index'] for item in chunks],
    'heading_path_json': [json.dumps(item['heading_path'], ensure_ascii=False) for item in chunks],
    'locator_json': [json.dumps(item['locator'], ensure_ascii=False, sort_keys=True) for item in chunks],
    'content_hash': [item['content_hash'] for item in chunks],
    'embedding_input_hash': embedding_input_hashes,
    'effective_from': [item['effective_from'] for item in chunks],
    'effective_to': [item['effective_to'] for item in chunks],
    'allowed_product_codes_json': [json.dumps(item['allowed_product_codes']) for item in chunks],
    'allowed_agent_scopes_json': [json.dumps(item['allowed_agent_scopes']) for item in chunks],
    'embedding_model': [MODEL_ID] * len(chunks),
    'embedding_revision': [resolved_revision] * len(chunks),
    'embedding_dimension': [DIMENSION] * len(chunks),
    'normalized': [NORMALIZE] * len(chunks),
    'embedding': embedding_column,
})

output_path = OUTPUT_ROOT / job['output_filename']
pq.write_table(table, output_path, compression='zstd')
parquet_hash = sha256_file(output_path)

chunk_hash_digest = sha256_text('\n'.join(
    f"{item['chunk_id']}|{item['content_hash']}" for item in chunks
))
generated_at = datetime.now(timezone.utc).isoformat()
dependencies = {
    name: package_version(name)
    for name in ['sentence-transformers', 'transformers', 'huggingface-hub', 'pyarrow', 'numpy', 'torch']
}

manifest = {
    'manifest_version': '1.0.0',
    'generated_at': generated_at,
    'index_tier': job['index_tier'],
    'model_id': MODEL_ID,
    'requested_revision': REQUESTED_REVISION,
    'resolved_revision': resolved_revision,
    'embedding_dimension': DIMENSION,
    'similarity': job['similarity'],
    'normalize_embeddings': NORMALIZE,
    'output_dtype': job['output_dtype'],
    'max_sequence_length': MAX_SEQUENCE_LENGTH,
    'batch_size': used_batch_size,
    'input_template_version': job['input_template_version'],
    'input_template': job['input_template'],
    'query_instruction': QUERY_INSTRUCTION,
    'parser_versions': sorted({item['parser_version'] for item in chunks}),
    'chunker_versions': sorted({item['chunker_version'] for item in chunks}),
    'source_registry_sha256': f'sha256:{sha256_file(sources_path)}',
    'chunk_file_sha256': f'sha256:{sha256_file(chunks_path)}',
    'chunk_content_hash_digest': f'sha256:{chunk_hash_digest}',
    'chunk_count': len(chunks),
    'source_count': len(sources),
    'dependencies': dependencies,
    'output': {
        'filename': output_path.name,
        'sha256': f'sha256:{parquet_hash}',
        'rows': table.num_rows,
    },
}

report = {
    'report_version': '1.0.0',
    'generated_at': generated_at,
    'status': 'PASS',
    'checks': {
        'rows': len(chunks),
        'duplicate_chunk_ids': len(chunk_ids) - len(set(chunk_ids)),
        'missing_source_records': len(missing_sources),
        'all_vectors_finite': bool(np.isfinite(vectors).all()),
        'vector_dimension': DIMENSION,
        'norm_min': float(norms.min()),
        'norm_mean': float(norms.mean()),
        'norm_max': float(norms.max()),
    },
}

(OUTPUT_ROOT / 'embedding-manifest.json').write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2) + '\n', encoding='utf-8'
)
(OUTPUT_ROOT / 'embedding-run-report.json').write_text(
    json.dumps(report, ensure_ascii=False, indent=2) + '\n', encoding='utf-8'
)

print({
    'output': str(output_path),
    'rows': table.num_rows,
    'sha256': parquet_hash,
    'bytes': output_path.stat().st_size,
})

In [ ]:
smoke_queries = [
    'Điều kiện vay vốn để bổ sung vốn lưu động là gì?',
    'Ngân hàng được xử lý dữ liệu cá nhân của khách hàng trong trường hợp nào?',
    'Khoản vay không có tài sản bảo đảm được quy định như thế nào?',
]

for query in smoke_queries:
    query_input = f'Instruct: {QUERY_INSTRUCTION}\nQuery: {query}'
    query_vector = model.encode(
        [query_input],
        convert_to_numpy=True,
        normalize_embeddings=NORMALIZE,
    )[0].astype(np.float32)
    scores = vectors @ query_vector
    top_indices = np.argsort(-scores)[:5]
    print(f'\nQUERY: {query}')
    for rank, index in enumerate(top_indices, start=1):
        chunk = chunks[int(index)]
        preview = chunk['content'].replace('\n', ' ')[:180]
        print(f"{rank}. {scores[index]:.4f} | {chunk['chunk_id']} | {preview}")

## Completion gate

Download all three files from `/kaggle/working`. Import is allowed only when `embedding-run-report.json` has `status: PASS`, the manifest revision is immutable, and the Parquet SHA-256 matches the manifest. This artifact remains experimental while its source registry is `IN_REVIEW`.